In [ ]:
import pandas as pd
import json
import os
from datetime import datetime
from firecrawl import FirecrawlApp
from pydantic import BaseModel, Field
from typing import List, Dict
import time

# Initialize Firecrawl app
app = FirecrawlApp(api_key="fc-c1d9edc656b24668a9ffc8c26ec17e60")

class ExtractSchema(BaseModel):
    # Ethnobotanical / geographic metadata
    scientific_name: str
    local_name: str
    local_name_citation: str
    family: str
    genus: str
    growth_form: str
    location_found_nepal: str
    location_found_nepal_citation: str
    traditional_usage: str
    traditional_usage_citation: str
    cultural_consensus_icf: float

    # Extraction / preparation
    preparation_solvent: str
    preparation_method: str
    preparation_method_citation: str
    extraction_solvent: str
    compound_class: str
    compound_class_citation: str
    parts_with_amr: str
    parts_with_amr_citation: str

    # Antimicrobial data
    pathogen_data: List[Dict] = Field(
        ..., description="List of pathogens with MIC, MBC, ZOI, assay type, extraction, preparation, and source URLs"
    )

# Configuration
CSV_FILE = "plant_list.csv"
OUTPUT_FILE = "plant_extraction_results.json"
PROCESSED_FILE = "processed_plants.txt"  # Track processed plant names
BATCH_SIZE = 5  # Number of plants to process before showing checkpoint
DELAY_BETWEEN_REQUESTS = 2  # Seconds to wait between API calls

def load_processed_plants():
    """Load list of already processed plant names from file"""
    if os.path.exists(PROCESSED_FILE):
        with open(PROCESSED_FILE, 'r', encoding='utf-8') as f:
            # Read and strip each line, filter out empty lines
            return set(line.strip() for line in f if line.strip())
    return set()

def save_processed_plant(plant_name):
    """Append a plant name to the processed plants file"""
    with open(PROCESSED_FILE, 'a', encoding='utf-8') as f:
        f.write(f"{plant_name}\n")
    print(f"✓ Added '{plant_name}' to processed list")

def load_existing_results():
    """Load existing results from JSON file if it exists"""
    if os.path.exists(OUTPUT_FILE):
        try:
            with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
                return json.load(f)
        except json.JSONDecodeError as e:
            print(f"⚠ Warning: Corrupted JSON file detected: {e}")
            print("Starting fresh with empty results")
            # Rename corrupted file
            corrupted_name = f"plant_extraction_results_corrupted_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
            os.rename(OUTPUT_FILE, corrupted_name)
            print(f"Renamed corrupted file to: {corrupted_name}")
    
    return {"extraction_date": datetime.now().isoformat(), "plants": [], "processed_count": 0}

def save_results(results):
    """Save results to JSON file"""
    results["last_updated"] = datetime.now().isoformat()
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"✓ Saved results to {OUTPUT_FILE}")

def extract_plant_data(plant_name):
    """Extract data for a single plant"""
    try:
        print(f"\n{'='*80}")
        print(f"Processing: {plant_name}")
        print(f"{'='*80}")
        
        result = app.agent(
            schema=ExtractSchema,
            prompt=(
                f"Extract detailed pharmacological and antimicrobial data for {plant_name}. "
                f"Ethnobotanical and geographic metadata must be specific to Nepal, including "
                f"local name, family, genus, growth form, altitude range, location, distribution, "
                f"traditional usage, and cultural consensus (ICF). For antimicrobial data, source "
                f"global scientific studies for pathogens like S. aureus, P. aeruginosa, K. pneumoniae, "
                f"and E. coli. Each pathogen entry must include MIC, MBC, ZOI, assay type, extraction "
                f"solvent, preparation method, and source URL citation. If multiple values exist for "
                f"the same pathogen, create separate entries. Output should match the flat schema where "
                f"each field is directly accessible."
            ),
            model="spark-1-mini",
        )
        
        # Convert Pydantic model to dict with JSON serialization
        plant_data = {
            "plant_name": plant_name,
            "extraction_timestamp": datetime.now().isoformat(),
            "status": "success",
            "data": json.loads(result.model_dump_json())  # Convert to JSON-serializable dict
        }
        
        print(f"✓ Successfully extracted data for {plant_name}")
        return plant_data
        
    except Exception as e:
        print(f"✗ Error processing {plant_name}: {str(e)}")
        return {
            "plant_name": plant_name,
            "extraction_timestamp": datetime.now().isoformat(),
            "status": "error",
            "error": str(e)
        }

def process_plants_in_batches():
    """Main function to process all plants from CSV"""
    # Load plant list from CSV
    df = pd.read_csv(CSV_FILE)
    plant_column = df.columns[0]  # First column contains plant names
    # Strip whitespace from plant names
    plant_list = [str(name).strip() for name in df[plant_column].dropna().tolist()]
    
    print(f"Loaded {len(plant_list)} plants from {CSV_FILE}")
    
    # Load list of already processed plants
    processed_plants = load_processed_plants()
    print(f"Already processed: {len(processed_plants)} plants")
    print(f"Remaining: {len(plant_list) - len(processed_plants)} plants")
    
    # Load existing results
    results = load_existing_results()
    
    # Process plants
    batch_count = 0
    new_plants_in_session = 0
    
    for i, plant_name in enumerate(plant_list, 1):
        # Skip if already processed
        if plant_name in processed_plants:
            print(f"[{i}/{len(plant_list)}] Skipping {plant_name} (already processed)")
            continue
        
        print(f"\n[{i}/{len(plant_list)}] Processing plant: {plant_name}")
        
        # Extract data
        plant_data = extract_plant_data(plant_name)
        
        # Add to results
        results["plants"].append(plant_data)
        results["processed_count"] = len(results["plants"])
        
        # Save results to JSON
        save_results(results)
        
        # Mark plant as processed (append to processed file)
        save_processed_plant(plant_name)
        processed_plants.add(plant_name)  # Update in-memory set
        
        batch_count += 1
        new_plants_in_session += 1
        
        # Add delay between requests to avoid rate limiting
        if i < len(plant_list):
            print(f"Waiting {DELAY_BETWEEN_REQUESTS} seconds before next request...")
            time.sleep(DELAY_BETWEEN_REQUESTS)
        
        # Optional: Save batch progress
        if batch_count % BATCH_SIZE == 0:
            print(f"\n{'#'*80}")
            print(f"BATCH CHECKPOINT: Processed {new_plants_in_session} new plants in this session")
            print(f"Total plants ever processed: {len(processed_plants)}")
            print(f"Total plants in database: {results['processed_count']}")
            print(f"{'#'*80}\n")
    
    print(f"\n{'='*80}")
    print(f"EXTRACTION COMPLETE!")
    print(f"New plants processed in this session: {new_plants_in_session}")
    print(f"Total plants ever processed: {len(processed_plants)}")
    print(f"Results saved to: {OUTPUT_FILE}")
    print(f"Processed list saved to: {PROCESSED_FILE}")
    print(f"{'='*80}")
    
    return results

# Start processing
print("Starting batch plant extraction...")
print(f"Configuration:")
print(f"  - CSV File: {CSV_FILE}")
print(f"  - Output File: {OUTPUT_FILE}")
print(f"  - Processed Plants File: {PROCESSED_FILE}")
print(f"  - Batch Size: {BATCH_SIZE}")
print(f"  - Delay Between Requests: {DELAY_BETWEEN_REQUESTS}s")
print()

results = process_plants_in_batches()

Starting batch plant extraction...
Configuration:
  - CSV File: plant_list.csv
  - Output File: plant_extraction_results.json
  - Batch Size: 5
  - Delay Between Requests: 2s

Loaded 1034 plants from plant_list.csv
Already processed: 0 plants
Remaining: 1034 plants

[1/1034] Processing plant: Acilepis squarrosa

Processing: Acilepis squarrosa


KeyboardInterrupt: 

In [ ]:
# View summary of results
with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f"Total plants processed: {data['processed_count']}")
print(f"Last updated: {data.get('last_updated', 'N/A')}")
print(f"\nSuccess rate:")
success_count = sum(1 for p in data['plants'] if p['status'] == 'success')
error_count = sum(1 for p in data['plants'] if p['status'] == 'error')
print(f"  ✓ Successful: {success_count}")
print(f"  ✗ Failed: {error_count}")

# Show first few plants
print(f"\nFirst 5 processed plants:")
for plant in data['plants'][:5]:
    status_icon = "✓" if plant['status'] == 'success' else "✗"
    print(f"  {status_icon} {plant['plant_name']} - {plant['status']}")

In [ ]:
# Optional: View detailed data for a specific plant
plant_name_to_view = "Swertia chirata"  # Change this to any plant name

with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
    data = json.load(f)

plant_data = next((p for p in data['plants'] if p['plant_name'] == plant_name_to_view), None)

if plant_data:
    print(f"Data for {plant_name_to_view}:")
    print(json.dumps(plant_data, indent=2))
else:
    print(f"Plant '{plant_name_to_view}' not found in results")